In [1]:
!pip install ~/fwiVis/utility_functions/
! pip install plotnine

Processing /home/jovyan/fwiVis/utility_functions
  Preparing metadata (setup.py) ... done
  Created wheel for fwiVis: filename=fwiVis-0.1-py3-none-any.whl size=17892 sha256=08f6429d671e398ffa27777d68d24e04e1a03f787c9600c7ca8e6ec243718961
  Stored in directory: /tmp/pip-ephem-wheel-cache-fp_jn55m/wheels/79/0f/3d/08c18473dd7e0fb915900e6b4f13b81f1fa84371f9bf24d864
Successfully built fwiVis
  Using cached plotnine-0.14.5-py3-none-any.whl.metadata (9.3 kB)
  Using cached mizani-0.13.1-py3-none-any.whl.metadata (4.7 kB)
Using cached plotnine-0.14.5-py3-none-any.whl (1.3 MB)
Using cached mizani-0.13.1-py3-none-any.whl (127 kB)


In [2]:
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
from math import cos, asin, sqrt
import re

import numpy as np
import geopandas as gpd
import pandas as pd
from matplotlib import pyplot as plt
import os
import rioxarray as rio
import xarray as xr
import rasterio
import glob
from shapely.errors import ShapelyDeprecationWarning
from shapely.geometry import Point
import warnings
import folium
import datetime
import time
from folium import plugins
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 
#import contextily as cx
from shapely.geometry import box
import sys
from datetime import datetime, timedelta
from itertools import chain

from datetime import date
from bs4 import BeautifulSoup
import requests
import os
import plotnine
import xarray as xr

import fwiVis.fwiVis as fv

In [3]:

# import fwiVis.fwiVis as fv

# # path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_only/April_1_unmerged_fires_with_FWI.csv"
# # path = "/projects/old_shared/fire_weather_vis/Lightning_analysis/fwi_timeline_merge/Final_dataset_as_of_20240209.csv"
# # fire3 = fv.prep_fire_files(path)

# path = os.path.abspath("data/Quebec_v3_full_data_perimeters20241112.csv")
# fire3 = fv.prep_fire_files(path)

# #ciffc = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/CIFFC_data/ciffc_all_canada.csv")
# ciffc = pd.read_csv(os.path.abspath("contextual_data/CIFFC_data/ciffc_all_canada.csv"))
# ciffc = ciffc[ciffc.field_agency_code == "qc"]

# ciffc = gpd.GeoDataFrame(ciffc, geometry= gpd.points_from_xy(ciffc.field_longitude, ciffc.field_latitude), crs = "4326")
# ciffc = ciffc.to_crs("3571")

# ciffc["geometry_point"] = ciffc.geometry

In [4]:
# fire3 = fire3.sort_values(by = ["fireID", "t"])
# fire3 = fire3[~fire3.FWI.isna()]
# #fire3.farea = fire3.farea.astype("int64")
# fire3 = fire3.sjoin(ciffc)
# fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()

# row_mask = (~fire3.fireID.str.contains("_"))

# #fire3[row_mask].hvplot.scatter(x='FWI', y='farea_diff', hover_cols=['fireID', 't'])
# #fire3[row_mask].plot.scatter(x='FWI', y='farea_diff')

In [5]:
# supression = fire3.groupby("fireID").field_response_type.unique().reset_index()
# supression

# supression["len"] = supression.field_response_type.apply(len)

# multi_suppress_ids = supression[supression.len> 1].fireID

# supression[supression.len> 1]
# fire_ids_not_unique_supression = supression[supression.len> 1].fireID

In [6]:
# ### Function that takes a point and looks for the multi polygons that intersect with that point. 
# #It then assignes just the polygon portion of the multipolygon with the point's ID. 
# # At the end, I want to put in an ID where there are two supression stratagies, and I want to get two IDs out, one per supression stratagy: subset to just the portion where there is one supression strategy.  
# # I have found one fire so far (1082) where this wont work, because at least some probable fire that could/should be tied to one surpession strategy doesn't overlap with the point, even after the reported time, and by the time out polygon overlapped with the point it was already merged with anouther one. So boo. 

# import shapely
# import shapely.geometry as gmt
# from shapely.geometry import MultiPolygon, Point, Polygon

# def split_multipolygon(multipolygon, point):
#     """Splits a MultiPolygon into separate polygons based on intersection with a point.

#     Args:
#         multipolygon (MultiPolygon): The MultiPolygon to split.
#         point (Point): The point used to determine intersection.

#     Returns:
#         list: A list of individual polygons.
#     """

#     polygons = []
#     for polygon in multipolygon:
#         if polygon.intersects(point):
#             polygons.append(polygon)
#     return polygons


# def exactly_one_true(lst):
#     """Checks if exactly one element in the list is True."""
#     return sum(lst) == 1

# def sep_supression(fid, df1,  meter_crs = 3571, point_crs = 4326): #df2,
#     print("Assuming df1 is in  crs 4326, and df2 is in a metered crs (35")
#     df = df1[df1.fireID == fid]
#     #df2 = df2[df2.fireID == fid]
#     #df = df[30:40]
#     # df = df.to_crs(meter_crs)
#     # df.geometry = df.geometry.buffer(500) ## 200 meeters
#     # df = df.to_crs(point_crs)
#     ciffc_resp = df.field_response_type.unique()
#     resp_l = []
#     #for r in ciffc_resp:
#     just_intersect = []
#     #unique_points = df[["field_latitude", "field_longitude", "field_response_type", "field_agency_fire_id"]].drop_duplicates().reset_index(drop = True) # (df.field_response_type == r)
#     #unique_points = gpd.GeoDataFrame(unique_points, geometry= gpd.points_from_xy(unique_points.field_longitude, unique_points.field_latitude,  crs = 4326))

#     unique_points = df[["geometry_point", "field_response_type", "field_agency_fire_id"]].drop_duplicates().reset_index(drop = True)
#     unique_points = gpd.GeoDataFrame(unique_points, geometry= unique_points.geometry_point,  crs = 3571)
#     #return(unique_points)
#     #points = [Point(lon, lat) for lon, lat in zip(unique_points.field_longitude, unique_points.field_latitude)]
#     points = unique_points.geometry_point
#     #print(len(points))
#     #return(points)
    
#     for index, row in df.iterrows(): # [df.field_response_type == r]
#     #if isinstance(row['geometry'], MultiPolygon):
#         #print("This is a multipolygon")
#        #split_polygons = split_multipolygon(row['geometry'], intersection_point)
#         ## Double check that only 1 point intersects with polygon. 
#         if(isinstance(row['geometry'], Polygon)):
#             row['geometry'] = gmt.MultiPolygon([row['geometry']])
#         for polygon in row['geometry'].geoms:
#             #print(shapely.intersects(points, polygon))
#             unique_points["Does_it_intersect"] = np.nan
#             intersecting_points = []
#             for point in points:
#                 intersecting_points.append(polygon.intersects(point))
                    
#             unique_points["Does_it_intersect"] = intersecting_points
#             up_group = unique_points.groupby("field_response_type").Does_it_intersect.unique().reset_index()
#             #return(up_group)

#             only_one_type_of_supression_intersects = (sum(up_group.Does_it_intersect.explode().values) == 1)
#             #intersecting_points = [point for point in points if polygon.intersects(point)]
#             #print(intersecting_points)

#             #if len(intersecting_points) == 1:  # Exactly one point intersects
#             if only_one_type_of_supression_intersects: ### Only one catagory intersects with this polygon
#                 #print(only_one_type_of_supression_intersects)
#                 #return(up_group)
#                 new_row = row.copy()
#                 new_row["geometry"] = polygon
#                 new_row["farea"] = (polygon.area / (1000 * 1000))
#                 new_row["fperim"] = np.nan
#                 new_row["meanFRP"] = np.nan
                
                
                
#                 # try:
#                     #new_row["fireID"] = str(fid) + "." +str(up_group[up_group.Does_it_intersect.sum()].field_response_type.iloc[0])# tmp2[tmp2.source.explode()].degree.iloc[0]
#                     #new_row["fireID"] = str(fid) + "." +str(up_group[up_group.Does_it_intersect.apply(any)].field_response_type.iloc[0]) + "." + str(*unique_points[unique_points.Does_it_intersect == True].field_agency_fire_id.unique())
#                 sup = str(up_group[up_group.Does_it_intersect.apply(any)].field_response_type.iloc[0])
#                 new_row["fireID"] = str(fid) + "." + sup +"." +  ".".join(unique_points[unique_points.Does_it_intersect == True].field_agency_fire_id.astype("str"))
#                 new_row["field_response_type"] = sup
                    
#                 # except:
#                 #     return(unique_points)
#                 # #     return(up_group)
                    
#                 just_intersect.append(new_row)
#                 # fig, ax = plt.subplots(figsize=(8, 6))
#                 # #print(type(new_row))
#                 # #new_row.plot(color = "green")
#                 # plt.plot(*new_row["geometry"].exterior.xy, color = "green")
#                 # for point in points:
#                 #     ax.plot(point.x, point.y, 'ro', label="Point")  # 'ro' for red points
                
#                 #     # Adjust the plot
#                 #     ax.set_title("Geometry and Points")
#                 #     ax.set_xlabel("Longitude")
#                 #     ax.set_ylabel("Latitude")
#                 #     plt.grid(True)
                    
#                 #     # Show the plot
#                 #     plt.show()
#                 #plt.plot(*new_row["geometry"].exterior.xy, color = "green")
#                 #plt.plot(*unique_points["geometry"].exterior.xy, color = "red")
#                 #unique_points.plot(color = "red")
#                 #plt.plot(unique_points.field_longitude, unique_points.field_latitude,  color = "red")
#                 #plt.show()
#             else:
#                 new_row = row.copy()
#                 new_row["geometry"] = polygon
#                 # fig, ax = plt.subplots(figsize=(8, 6))
#                 # plt.plot(*new_row["geometry"].exterior.xy, color = "yellow")
#                 # for point in points:
#                 #     ax.plot(point.x, point.y, 'ro', label="Point")  # 'ro' for red points
                
#                 #     # Adjust the plot
#                 #     ax.set_title("Geometry and Points")
#                 #     ax.set_xlabel("Longitude")
#                 #     ax.set_ylabel("Latitude")
#                 #     plt.grid(True)
                    
#                 #     # Show the plot
#                 #     plt.show()
#                 #plt.plot(*new_row["geometry"].exterior.xy, color = "yellow")
#                 #plt.plot(unique_points.field_longitude, unique_points.field_latitude,  color = "red")
#                 #plt.plot(*unique_points["geometry"].exterior.xy, color = "red")
#                 #unique_points.plot(color = "red")
#                 #plt.show()
#                 #print("skipping")
#                 #just_intersect.append(None)
#     # else:
#     #     just_intersect.append(row)
#     just_intersect_df = gpd.GeoDataFrame(just_intersect)
#     just_intersect_df = gpd.GeoDataFrame(just_intersect_df, geometry = just_intersect_df.geometry, crs = 3571)
#     #just_intersect_df = just_intersect_df[just_intersect_df.field_response_type == sup] # Dropping false lable
#     just_intersect_df = just_intersect_df.drop_duplicates()
#     # if(len(just_intersect_df) > 0):
#     #     #print(just_intersect)
#     #     just_intersect_df.fireID = just_intersect_df.fireID.astype("str") + "." + r
#     #     #print(just_intersect_df)
#     #     resp_l.append(just_intersect_df)
#     # else:
#     #     print(f"{fid} had no independant multi-polygons to split")
# #full_df = pd.concat(resp_l, axis=0)
#     return(just_intersect_df)
    

# def check_that_continious_record(df):
#     min_t = df.t.min()
#     max_t = df.t.max()
#     #dates = pd.date_range(start= min_t, end=  max_t).to_pydatetime().tolist()
#     dates = pd.date_range(start= min_t, end=  max_t).strftime('%Y-%m-%d 12:00:00').tolist()
#     len_df = len(df.t.unique())
#     #print(df.t.unique())
#     len_seq = len(dates)
#     #print(dates)
#     if(len_df != len_seq):
#         print(f"FireID {df.fireID.unique()} is not a continious sequence.")
#         return(False)
#     d_list = []
#     for d in dates:
#         some_dates = df[df.t == d]
#         d_list.append(len(some_dates) > 0)
#     return(any(d_list))


    

In [7]:
# ### remake IDs with duel supression status 


# skip_list = ["1082", 
#             "1324"] ### has more than one supression, but seems like the reported point doesn't overlap with an indepandant multipolygon. 

# duel_sup =  [item for item in fire_ids_not_unique_supression if item not in skip_list]
# dfs = []
# for s in duel_sup:
#     print(s)
#     tmp = sep_supression(str(s), fire3)
#     dfs.append(tmp)
# foo = pd.concat(dfs)

In [8]:
# foo = gpd.GeoDataFrame(foo, geometry = foo.geometry, crs = 3571)
# #foo = foo.set_crs(4326)

In [9]:
# #### Check that there is a continious t 


# bools = foo.groupby("fireID").apply(check_that_continious_record)
# print(f"Are all the fires continuios records???: {all(bools)}")

# if(all(bools)):
#     fire3 = fire3[~fire3.fireID.isin(fire_ids_not_unique_supression)]
#     foo = foo.to_crs(fire3.crs)
#     fire3 = pd.concat([fire3, foo], ignore_index=True)

In [11]:
fire3 = pd.read_csv(f"{os.path.abspath("landcover_merged_datasets")}/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")

In [12]:
fire3 = fire3.sort_values(by = ["fireID", "t"])
fire3 = fire3[~fire3.FWI.isna()]

fire_export = fv.prep_fire_files(f"{os.path.abspath("landcover_merged_datasets")}/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv", 4326)
fire_export["lon_centroid"] = fire_export["geometry"].centroid.x
fire_export["lat_centroid"] = fire_export["geometry"].centroid.y
fire_export.to_csv(f"{os.path.abspath("landcover_merged_datasets")}/with_centroids_fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")

/tmp/ipykernel_1019/1490796534.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

/tmp/ipykernel_1019/1490796534.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



In [ ]:
fire

In [12]:
#import numpy as np
from matplotlib import pyplot as plt
from plotnine import ggplot, geom_point, geom_jitter, aes, stat_smooth, facet_wrap
import plotnine as plotnine




### Some useful vars to color by 
def assign_day_of_fire(df):
    df = df.sort_values(by = "t")
    df['day_of_fire'] = df.t.rank()
    #df['day_of_fire'] = df['day_of_fire'].astype("int64")
    return(df)


def get_max_duration(df):
    max_duration = df.duration.max()
    df["max_duration"] = max_duration


fire3["FWI_diff"] = fire3.groupby("fireID").FWI.diff()
fire3["farea_diff"] = fire3.groupby("fireID").farea.diff()


fire3 = fire3.groupby("fireID").apply(assign_day_of_fire).reset_index(drop = True)
fire3["farea_shifted"] = fire3.groupby("fireID").farea.shift(periods = 1)

fire3["normalized_farea_diff"] = fire3.farea_diff/fire3.farea_shifted

rolling_num = 3
agg_function = "max" # max

fire3["FWI_rolling"] = fire3.groupby("fireID").FWI.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
# fire3["FWI_norm_rolling"] = fire3.groupby("fireID").FWI_norm.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["FWI_diff_rolling"] = fire3.groupby("fireID").FWI_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3['farea_rolling'] = fire3.groupby("fireID").farea.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3['farea_diff_rolling'] = fire3.groupby("fireID").farea_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3["GEOS-5.IMERGEARLY_rolling"] = fire3.groupby("fireID")["GEOS-5.IMERGEARLY"].rolling(rolling_num).agg(agg_function).reset_index(drop = True)

agg_function = "mean" # max
fire3[f"FWI_rolling_{agg_function}"] = fire3.groupby("fireID").FWI.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3[f"FWI_diff_rolling_{agg_function}"] = fire3.groupby("fireID").FWI_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3[f'farea_rolling_{agg_function}'] = fire3.groupby("fireID").farea.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3[f'farea_diff_rolling_{agg_function}'] = fire3.groupby("fireID").farea_diff.rolling(rolling_num).agg(agg_function).reset_index(drop = True)
fire3[f"GEOS-5.IMERGEARLY_rolling_{agg_function}"] = fire3.groupby("fireID")["GEOS-5.IMERGEARLY"].rolling(rolling_num).agg(agg_function).reset_index(drop = True)

long_fires = fire3[fire3.day_of_fire > 3].fireID.unique()
#fire3['max_dof'] = fire3.groupby("fireID").day_of_fire.max()

row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires)) #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)


fire3["GEOS5_IMERGEARLY_rolling"] = fire3["GEOS-5.IMERGEARLY_rolling"]
fire3["GEOS5_IMERGEARLY"] = fire3["GEOS-5.IMERGEARLY"]
fire3["GEOS5_IMERGEARLY_rolling_mean"] = fire3["GEOS-5.IMERGEARLY_rolling_mean"]

fire3["log_farea_diff_rolling"] = np.log(fire3["farea_diff_rolling"] + 0.999)

#x_var = ["FWI","FWI_rolling", "FWI_diff_rolling", "FWI_norm_rolling"]
#y_var = ['farea', "normalized_farea_diff", 'farea_rolling', 'farea_diff_rolling']


# x_var = ["FWI","FWI_rolling", "FWI_diff_rolling", "GEOS-5.IMERGEARLY", "GEOS-5.IMERGEARLY_rolling"]
# y_var = [ 'farea_diff_rolling']

# for x in x_var:
#     for y in y_var:
#         #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = 'field_latitude'))
#         p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "field_response_type"))
#         #p = (ggplot(fire3[row_mask], aes( x = x, y = y, color = "farea_shifted"))
#          + geom_point()
#          + plotnine.labels.ylab(y)
#          + plotnine.labels.xlab(x)
#          + stat_smooth(method = "glm", formula = "y ~ x")
#          #+ plotnine.scale_y_log10()
#          + plotnine.ggtitle(f"{agg_function} in rolling window of {rolling_num} days")

#          )
#         p.show()
#         print(p)
#         #del(p)

/tmp/ipykernel_2521/732720103.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


## Master Stats function

In [13]:
def run_stats(df, x, y, c, opt_tag = "", index_vars = ['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"], verbose = False, plot = False):

    df = df[index_vars].dropna()
    
    

    df = df.sort_values(by = x)
    df = df[[y, x, c]].drop_duplicates()

    # # #Unique category labels: 'D', 'F', 'G', ...
    color_labels = df[c].unique()
    
    # # # List of RGB triplets
    
    rgb_values = sns.color_palette("colorblind", len(color_labels))
    # # #rgb_values.reverse()
    
    # # # Map label to RGB
    color_map = dict(zip(color_labels, rgb_values))
    
    formula = f"{y} ~ {x}:C({c})"
    # print(formula)
    
    families = ["Gaussian", "Gamma",  "NegativeBinomial"] # "Poisson" causing numerical issues
    links = ["Log", "Identity"]
    #links = ["Identity"]
    
    ls = []
    fs = []
    aic = []
    ll = []
    bic = []
    for f in families:
        for l in links: 
            #desc = ModelDesc.from_formula(formula)
            #desc.describe()
    
            #sm.families.family.Gamma.links
            #link_g = sm.genmod.families.links.Identity()
            #link_g = sm.genmod.families.links.Log()
            #link_g = sm.genmod.families.links.CLogLog()
            #link_g = sm.genmod.families.links.Sqrt()
            #link_g = sm.genmod.families.links.InversePower()
            #link_g = sm.genmod.families.links.NegativeBinomial()
            #link_g = sm.genmod.families.links.Power()
            
            link_g = getattr(sm.genmod.families.links, l)
    
            method = getattr(sm.families, f)
    
            model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
            if (verbose):
                print(model.summary())
            tmp = model.summary2()
            
            fs.append(f)
            ls.append(l)
            aic.append(model.aic)
            ll.append(tmp.tables[0].iloc[2,3])
            bic.append(model.bic)
    
    
    
            df['fitted'] = model.fittedvalues
            df['residuals'] = model.resid_response
            df = df.sort_values(by = x)
          
            # # Plot residuals vs fitted values
            # res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
            # res
            # plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
            # plt.xlabel('Fitted Values')
            # plt.ylabel('Residuals')
            # plt.show()
            
    
            predictions = model.get_prediction(df, transform = True) #df, transform = False
            df['predicted'] = predictions.predicted_mean
            df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T

            if (plot):
                
                actual = sns.scatterplot(x=x, y=y, data=df, hue = c, palette= color_map) # hue = c)
                actual
                #handles, labels = actual.get_legend_handles_labels()
        
                # Customize legend titles and labels
                #actual.legend(handles=handles, labels= [*color_labels], title= c)
                # Plot the fitted values
                pred = sns.lineplot(x=x, y='predicted', data=df, hue = c, palette= color_map, legend = False) #  hue =c,
                pred
                # Plot the confidence intervals
                for cat in df[c].unique():
                    #print(cat)
                    #print(df.loc[(df[c] == cat), [c]].map(color_map))
                    plt.fill_between(df.loc[(df[c] == cat)][x], df.loc[(df[c] == cat)]['conf_int_low'], df.loc[(df[c] == cat)]['conf_int_high'],  color=color_map[cat], alpha=0.3)
        
                plt.title(f'{f} Fit with {l} link')
                plt.xlabel("Fire Weather Index")
                plt.ylabel("Fire Area Growth km^2")
                #plt.legend()
                plt.savefig(f"{os.path.abspath('some_figs')}/model_fit_{f}_with_{l}_{y}_func_of_{x}_by_{c}_bic{model.bic}{opt_tag}.png", dpi = 900, transparent = False)
                plt.show()
            
    stats_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})


    #print(stats_fwi_rolling[stats_fwi_rolling.Log_Likelyhood.astype("float").min() == stats_fwi_rolling.Log_Likelyhood.astype("float") ])
    
    #print(stats_fwi_rolling[stats_fwi_rolling.AIC.astype("float").min() == stats_fwi_rolling.AIC.astype("float") ])
    #print(stats_fwi_rolling[stats_fwi_rolling.BIC.astype("float").min() == stats_fwi_rolling.BIC.astype("float") ])
    return(stats_fwi_rolling)


def run_stats_single(df, x, y, c, opt_tag = "", index_vars = ['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"], verbose = False, plot = False):

    df = df[index_vars].dropna()
    df = df.sort_values(by = x)
    df = df[[y, x]].drop_duplicates()
    
    formula = f"{y} ~ {x}"
    #print(formula)
    
    families = ["Gaussian", "Gamma",  "NegativeBinomial"]
    
    links = ["Log", "Identity"]
    
    ls = []
    fs = []
    aic = []
    ll = []
    bic = []
    for f in families:
        for l in links: 
            #desc = ModelDesc.from_formula(formula)
            #desc.describe()
    
            #sm.families.family.Gamma.links
            #link_g = sm.genmod.families.links.Identity()
            #link_g = sm.genmod.families.links.Log()
            #link_g = sm.genmod.families.links.CLogLog()
            #link_g = sm.genmod.families.links.Sqrt()
            #link_g = sm.genmod.families.links.InversePower()
            #link_g = sm.genmod.families.links.NegativeBinomial()
            #link_g = sm.genmod.families.links.Power()
            
            link_g = getattr(sm.genmod.families.links, l)
    
            method = getattr(sm.families, f)
    
            model = smf.glm(formula, data=df, family=method(link = link_g(), check_link=True)).fit() # family=sm.families.Poisson()
            if(verbose):  
                print(model.summary())
            tmp = model.summary2()
            
            fs.append(f)
            ls.append(l)
            aic.append(model.aic)
            ll.append(tmp.tables[0].iloc[2,3])
            bic.append(model.bic)
    
    
    
            df['fitted'] = model.fittedvalues
            df['residuals'] = model.resid_response
    
            # # Plot residuals vs fitted values
            # res = sns.residplot(x='fitted', y='residuals', data=df, lowess=True)
            # res
            # plt.title(f'{f} Fit with {l} link: Residuals vs Fitted Values')
            # plt.xlabel('Fitted Values')
            # plt.ylabel('Residuals')
            # plt.show()
            
    
            predictions = model.get_prediction(df, transform = True) #df, transform = False
            df['predicted'] = predictions.predicted_mean
            df['conf_int_low'], df['conf_int_high'] = predictions.conf_int().T
            df = df.sort_values(by = x)
            if(plot):
                actual = sns.scatterplot(x=x, y=y, data=df)
                actual
                # Plot the fitted values
                pred = sns.lineplot(x=x, y='predicted', data=df, legend = False)
                pred
                # Plot the confidence intervals
                #plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], c ="field_response_type", alpha=0.3)
           
                
                plt.fill_between(df['FWI_rolling'], df['conf_int_low'], df['conf_int_high'], alpha=0.3)
        
                plt.title(f'{f} Fit with {l} link')
                plt.xlabel(x)
                plt.ylabel(y)
                #plt.legend()
                plt.savefig(f'{os.path.abspath("some_figs/")}/model_fit_{f}_with_{l}_{y}_func_of_{x}_{model.bic}{opt_tag}.png', dpi = 900, transparent = False)
                plt.show()
    stats_single_model_fwi_rolling = pd.DataFrame({"Family": fs, "Link_Function" : ls,  "Log_Likelyhood": ll, "AIC": aic, "BIC": bic})
    return(stats_single_model_fwi_rolling)
        


## Test single-model function

In [14]:
### Trying to fit a glm for non-constant varience 
from patsy import ModelDesc
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf


row_mask = (~fire3.fireID.str.contains("_")) & (fire3.farea_diff > 0.1) & (fire3.fireID.isin(long_fires))  #& fire3['max_dof'] >= 3#& (fire3.max_duration >= 3)
#row_mask = (~fire3.fireID.str.contains("_")) 

tmp_lc = fire3.groupby('dominant_landcover').count().reset_index()
low_n_landcover = tmp_lc[tmp_lc.fireID <= 10].dominant_landcover.unique()

row_mask_fl = row_mask & ~fire3.dominant_landcover.isin(low_n_landcover)


index_vars = ['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
'field_response_type', 
'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]


# df = fire3[row_mask]



# df = df[['fireID', 't',  'GEOS-5.IMERGEARLY', 'FWI',
# 'field_response_type', 
# 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling',"GEOS5_IMERGEARLY_rolling", "log_farea_diff_rolling"]].dropna()


# y = "farea_diff_rolling"
# x = "GEOS5_IMERGEARLY_rolling"
# c = "field_response_type"




# imerg_sup = run_stats(df, x, y, c, opt_tag = "")

In [15]:
data = {
    'Model': ["y ~ x:(Suppression Response)","y ~ xIMERGE:(Suppression Response)", "y ~ x", "y ~ xIMERGE", "y ~ x:(Fuel Type)", "y ~ xIMERGE:(Fuel Type)", "yFuels N > 10 ~ xFuels N > 10:(Fuel Type)", "yFuels N > 10 ~ xIMERGEFuels N > 10:(Fuel Type)", "yFuels N > 10 ~ xFuels N > 10:(Suppression Response)"

, "yFuels N > 10 ~ xIMERGEFuels N > 10:(Suppression Response)"], 
    'y': ["farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling", "farea_diff_rolling"],
    'x': ['FWI_rolling', "GEOS5_IMERGEARLY_rolling" , 'FWI_rolling', "GEOS5_IMERGEARLY_rolling" , 'FWI_rolling', "GEOS5_IMERGEARLY_rolling" , 'FWI_rolling', "GEOS5_IMERGEARLY_rolling" , 'FWI_rolling', "GEOS5_IMERGEARLY_rolling" ],
    'c': ["field_response_type", "field_response_type", np.nan, np.nan, 'dominant_landcover', 'dominant_landcover', 'dominant_landcover', 'dominant_landcover',"field_response_type", "field_response_type"], 
    'func': [run_stats, run_stats, run_stats_single, run_stats_single, run_stats, run_stats, run_stats, run_stats, run_stats, run_stats],
    'row_mask': [row_mask, row_mask, row_mask, row_mask, row_mask, row_mask,row_mask_fl, row_mask_fl, row_mask_fl, row_mask_fl], 
    'opt_tag': ["", "", "_single_model", "_single_model", "", "", "fules_larger_10", "fules_larger_10", "fules_larger_10", "fules_larger_10"], 
    'index_vars': [index_vars, index_vars, index_vars, index_vars, [*index_vars, 'dominant_landcover' ], [*index_vars, 'dominant_landcover'],  [*index_vars, 'dominant_landcover' ], [*index_vars, 'dominant_landcover'], [*index_vars, 'dominant_landcover' ], [*index_vars, 'dominant_landcover']],
    'Distribution': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan], 
    'Link': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan], 
    'AIC': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan],
    'BIC': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan]
    
    
}
model_compare = pd.DataFrame(data)

## Loop through for all the models. 

In [16]:
for i, row in model_compare.iloc[0:3].iterrows():
    if callable(row["func"]):  
        model_fits = row["func"](df = fire3[row['row_mask']], y = row['y'],  x = row['x'], c = row['c'], opt_tag = row['opt_tag'], index_vars = row['index_vars'])
        model_fits = model_fits[model_fits.BIC == model_fits.BIC.min()]
        model_compare.loc[i, 'Distribution'] = model_fits.Family.iloc[0]
        model_compare.loc[i, 'Link'] = model_fits.Link_Function	.iloc[0]
        model_compare.loc[i, 'AIC'] = model_fits.AIC.iloc[0]
        model_compare.loc[i, 'BIC'] = model_fits.BIC.iloc[0]
    

/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress

## Render model comparison table

In [17]:
model_compare

,Model,y,x,c,func,row_mask,opt_tag,index_vars,Distribution,Link,AIC,BIC
0,y ~ x:(Suppression Response),farea_diff_rolling,FWI_rolling,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5943.240042,-2497.494228
1,y ~ xIMERGE:(Suppression Response),farea_diff_rolling,GEOS5_IMERGEARLY_rolling,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,5858.419536,-2495.724808
2,y ~ x,farea_diff_rolling,FWI_rolling,NaN,<function run_stats_single at 0x7f6e00589e40>,0 False 1 True 2 False 3 ...,_single_model,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5963.455940,-2486.137769
3,y ~ xIMERGE,farea_diff_rolling,GEOS5_IMERGEARLY_rolling,NaN,<function run_stats_single at 0x7f6e00589e40>,0 False 1 True 2 False 3 ...,_single_model,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN
4,y ~ x:(Fuel Type),farea_diff_rolling,FWI_rolling,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN
5,y ~ xIMERGE:(Fuel Type),farea_diff_rolling,GEOS5_IMERGEARLY_rolling,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN
6,yFuels N > 10 ~ xFuels N > 10:(Fuel Type),farea_diff_rolling,FWI_rolling,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN
7,yFuels N > 10 ~ xIMERGEFuels N > 10:(Fuel Type),farea_diff_rolling,GEOS5_IMERGEARLY_rolling,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN
8,yFuels N > 10 ~ xFuels N > 10:(Suppression Res...,farea_diff_rolling,FWI_rolling,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN
9,yFuels N > 10 ~ xIMERGEFuels N > 10:(Suppressi...,farea_diff_rolling,GEOS5_IMERGEARLY_rolling,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NaN,NaN,NaN,NaN


In [18]:
# fire = fire.reset_index()


# def rasterize_geometry(geometry, xarray_data):
#     transform = xarray_data.rio.transform()
#     out_shape = xarray_data.rio.shape
#     return features.geometry_mask([mapping(geometry)], transform=transform, invert=True, out_shape=out_shape)

# # Initialize a list to store the dominant landcover types
# dominant_landcovers = []

# # Loop through each geometry in the GeoDataFrame
# itr = 0
# for i in fire.index:
#     print(i)
#     geometry = fire.iloc[i].geometry
#     rm = (fire.geometry == geometry)
#     itr = itr + 1
#     print(itr/len(fire))
#     # clip fuels to geometry bounds
#     if(len(fire[rm].geometry.unique()) > 1):
#         raise ValueError("There are multiple geometries to get coordinates from. ")

#     min_y = round(fire[rm].bounds.miny.min(),1)
#     max_y = round(fire[rm].bounds.maxy.max(), 1)

#     min_x = round(fire[rm].bounds.minx.min(),1)
#     max_x = round(fire[rm].bounds.maxx.max(), 1)

#     # min_y = fire[rm].bounds.miny.min()
#     # max_y = fire[rm].bounds.maxy.max()

#     # min_x = fire[rm].bounds.minx.min()
#     # max_x = fire[rm].bounds.maxx.max()
    

#     # print(min_y, max_y, min_x, max_x)
    
#     if(((max_x - min_x) <=3 ) | ((max_y - min_y) <= 3 )):
#         max_x = max_x + 5
#         max_y = max_y + 5

#     fuels_sm = fuels[landcover_vars[0]]
#     # try:
#     fuels_sm = fuels_sm.sel(lon=slice(min_x, max_x), 
#                lat=slice(  max_y, min_y))

#     # print(min_y, max_y, min_x, max_x)
#     # # except:
#     # has_data = fuels_sm.isnull().all()
#     # if not has_data:
#     # #if (fuels_sm.time.size == 0):
#     #     raise ValueError("Subset contains no data; adjust slicing bounds.")

    
#     # Rasterize the geometry to create a mask
#     # try:
#     mask = rasterize_geometry(geometry, fuels_sm)
#     # mask = rasterize_geometry(geometry, fuels)
#     # except:
    
    


    
#     #mask = rasterize_geometry(geometry, fuels[landcover_vars[0]])
#     # except:
#     #     print(min_y, max_y, min_x, max_x)


  
    
#     # Initialize a dictionary to store the sum of each landcover type within the geometry
#     landcover_sums = {var: 0 for var in landcover_vars}
    
#     # Loop through each landcover variable
#     for var in landcover_vars:
#         # Access the DataArray corresponding to the current landcover variable
#         #data_array = fuels[var]
#         #data_array = fuels_sm[var]
#         data_array = fuels[var].sel(lon=slice(min_x, max_x), 
#                lat=slice(  max_y, min_y))
        
#         # Apply the mask to the current landcover data variable using xr.where
#         masked_landcover = data_array.where(mask)
        
#         # Sum the values within the masked area and store in the dictionary
#         #landcover_sums[var] = masked_landcover.sum().item()
#         landcover_sums[var] = masked_landcover.sum().compute().item()
    
#     # Determine the dominant landcover type (the one with the highest sum)
#     dominant_landcover = max(landcover_sums, key=landcover_sums.get)
#     dominant_landcovers.append(dominant_landcover)

# # Add the dominant landcover types to the GeoDataFrame
# fire['dominant_landcover'] = dominant_landcovers

# # Save or further process the GeoDataFrame
# #gdf.to_file('path_to_save_geodataframe.shp')

# print(fire[['geometry', 'dominant_landcover']])

# # def get_dominant_value(geometry, raster):
# #     # Clip the raster with the geometry
# #     clipped = raster.rio.clip([geometry])
    
# #     # Get the raster values within the geometry
# #     values = clipped.values.flatten()
    
# #     # Remove NaN values
# #     values = values[~np.isnan(values)]
    
# #     # Find the most common value
# #     if len(values) > 0:
# #         most_common_value = Counter(values).most_common(1)[0][0]
# #     else:
# #         most_common_value = np.nan
    
# #     return most_common_value

# # # # Apply the function to each geometry in the vector dataset
# # fire['dominant_raster_value'] = fire.geometry.apply(lambda geom: get_dominant_value(geom, fuels))

# # #gf.sel(lon = slice(round(fire.bounds.minx.min(),1), round(fire.bounds.maxx.max(),1)), lat = slice(46, 58))#.mean(dim='time').plot()

In [19]:
#fuels_sm[landcover_vars[0]].rio.transform()

In [20]:
#len(fire.dominant_landcover)

In [21]:
#len(fire.field_response_type)

In [22]:
#fire.to_csv(f"{os.path.abspath("landcover_merged_datasets")}/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")

In [23]:
#fire.geometry[54]

In [24]:
#fire = pd.read_csv("/projects/old_shared/fire_weather_vis/Lightning_analysis/landcover_merged_datasets/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")
fire = pd.read_csv(f"{os.path.abspath("landcover_merged_datasets")}/fires_merged_ESACCI-LC-L4-PFT-Map-300m-P1Y-2020-v2.0.8.csv")

# Double check if no smoothing has same patterns

In [25]:
data = {
    'Model': ["y ~ x:(Suppression Response)","y ~ xIMERGE:(Suppression Response)", "y ~ x", "y ~ xIMERGE", "y ~ x:(Fuel Type)", "y ~ xIMERGE:(Fuel Type)", "yFuels N > 10 ~ xFuels N > 10:(Fuel Type)", "yFuels N > 10 ~ xIMERGEFuels N > 10:(Fuel Type)", "yFuels N > 10 ~ xFuels N > 10:(Suppression Response)"

, "yFuels N > 10 ~ xIMERGEFuels N > 10:(Suppression Response)"], 
    'y': ["farea_diff", "farea_diff", "farea_diff", "farea_diff", "farea_diff", "farea_diff", "farea_diff", "farea_diff", "farea_diff", "farea_diff"],
    'x': ['FWI', "GEOS5_IMERGEARLY" , 'FWI', "GEOS5_IMERGEARLY" , 'FWI', "GEOS5_IMERGEARLY" , 'FWI', "GEOS5_IMERGEARLY" , 'FWI', "GEOS5_IMERGEARLY" ],
    'c': ["field_response_type", "field_response_type", np.nan, np.nan, 'dominant_landcover', 'dominant_landcover', 'dominant_landcover', 'dominant_landcover',"field_response_type", "field_response_type"], 
    'func': [run_stats, run_stats, run_stats_single, run_stats_single, run_stats, run_stats, run_stats, run_stats, run_stats, run_stats],
    'row_mask': [row_mask, row_mask, row_mask, row_mask, row_mask, row_mask,row_mask_fl, row_mask_fl, row_mask_fl, row_mask_fl], 
    'opt_tag': ["", "", "_single_model", "_single_model", "", "", "fules_larger_10", "fules_larger_10", "fules_larger_10", "fules_larger_10"], 
    'index_vars': [[*index_vars, "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, 'dominant_landcover' ,  "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, 'dominant_landcover',  "farea_diff", "GEOS5_IMERGEARLY"],  [*index_vars, 'dominant_landcover' ,  "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, 'dominant_landcover',  "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, 'dominant_landcover' ,  "farea_diff", "GEOS5_IMERGEARLY"], [*index_vars, 'dominant_landcover',  "farea_diff", "GEOS5_IMERGEARLY"]],
    'Distribution': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan], 
    'Link': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan], 
    'AIC': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan],
    'BIC': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan]
    
    
}
model_compare_no_smooth = pd.DataFrame(data)

In [26]:
for i, row in model_compare_no_smooth.iterrows():
    if callable(row["func"]):  
        model_fits_no_smooth = row["func"](df = fire3[row['row_mask']], y = row['y'],  x = row['x'], c = row['c'], opt_tag = row['opt_tag'], index_vars = row['index_vars'])
        model_fits_no_smooth = model_fits_no_smooth[model_fits_no_smooth.BIC == model_fits_no_smooth.BIC.min()]
        model_compare_no_smooth.loc[i, 'Distribution'] = model_fits_no_smooth.Family.iloc[0]
        model_compare_no_smooth.loc[i, 'Link'] = model_fits_no_smooth.Link_Function	.iloc[0]
        model_compare_no_smooth.loc[i, 'AIC'] = model_fits_no_smooth.AIC.iloc[0]
        model_compare_no_smooth.loc[i, 'BIC'] = model_fits_no_smooth.BIC.iloc[0]

/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress

In [27]:
model_compare_no_smooth

,Model,y,x,c,func,row_mask,opt_tag,index_vars,Distribution,Link,AIC,BIC
0,y ~ x:(Suppression Response),farea_diff,FWI,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,6518.928090,-2789.592037
1,y ~ xIMERGE:(Suppression Response),farea_diff,GEOS5_IMERGEARLY,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,6435.158867,-2873.361260
2,y ~ x,farea_diff,FWI,NaN,<function run_stats_single at 0x7f6e00589e40>,0 False 1 True 2 False 3 ...,_single_model,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,6533.138134,-2784.506881
3,y ~ xIMERGE,farea_diff,GEOS5_IMERGEARLY,NaN,<function run_stats_single at 0x7f6e00589e40>,0 False 1 True 2 False 3 ...,_single_model,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,6467.051348,-2850.593667
4,y ~ x:(Fuel Type),farea_diff,FWI,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,6474.889319,-2824.505919
5,y ~ xIMERGE:(Fuel Type),farea_diff,GEOS5_IMERGEARLY,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,6369.837971,-2929.557267
6,yFuels N > 10 ~ xFuels N > 10:(Fuel Type),farea_diff,FWI,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,6474.889319,-2824.505919
7,yFuels N > 10 ~ xIMERGEFuels N > 10:(Fuel Type),farea_diff,GEOS5_IMERGEARLY,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,6369.837971,-2929.557267
8,yFuels N > 10 ~ xFuels N > 10:(Suppression Res...,farea_diff,FWI,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,6518.928090,-2789.592037
9,yFuels N > 10 ~ xIMERGEFuels N > 10:(Suppressi...,farea_diff,GEOS5_IMERGEARLY,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,6435.158867,-2873.361260


## Try mean as smoother

In [28]:
index_vars = [*index_vars, 'FWI_rolling_mean', "farea_diff_rolling_mean",  "GEOS5_IMERGEARLY_rolling_mean"]

In [29]:
data = {
    'Model': ["y ~ x:(Suppression Response)","y ~ xIMERGE:(Suppression Response)", "y ~ x", "y ~ xIMERGE", "y ~ x:(Fuel Type)", "y ~ xIMERGE:(Fuel Type)", "yFuels N > 10 ~ xFuels N > 10:(Fuel Type)", "yFuels N > 10 ~ xIMERGEFuels N > 10:(Fuel Type)", "yFuels N > 10 ~ xFuels N > 10:(Suppression Response)"

, "yFuels N > 10 ~ xIMERGEFuels N > 10:(Suppression Response)"], 
    'y': ["farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean", "farea_diff_rolling_mean"],
    'x': ['FWI_rolling_mean', "GEOS5_IMERGEARLY_rolling_mean" , 'FWI_rolling_mean', "GEOS5_IMERGEARLY_rolling_mean" , 'FWI_rolling_mean', "GEOS5_IMERGEARLY_rolling_mean" , 'FWI_rolling_mean', "GEOS5_IMERGEARLY_rolling_mean" , 'FWI_rolling_mean', "GEOS5_IMERGEARLY_rolling_mean" ],
    'c': ["field_response_type", "field_response_type", np.nan, np.nan, 'dominant_landcover', 'dominant_landcover', 'dominant_landcover', 'dominant_landcover',"field_response_type", "field_response_type"], 
    'func': [run_stats, run_stats, run_stats_single, run_stats_single, run_stats, run_stats, run_stats, run_stats, run_stats, run_stats],
    'row_mask': [row_mask, row_mask, row_mask, row_mask, row_mask, row_mask,row_mask_fl, row_mask_fl, row_mask_fl, row_mask_fl], 
    'opt_tag': ["", "", "_single_model", "_single_model", "", "", "fules_larger_10", "fules_larger_10", "fules_larger_10", "fules_larger_10"], 
    'index_vars': [index_vars, index_vars, index_vars, index_vars, [*index_vars, 'dominant_landcover' ], [*index_vars, 'dominant_landcover'],  [*index_vars, 'dominant_landcover' ], [*index_vars, 'dominant_landcover'], [*index_vars, 'dominant_landcover' ], [*index_vars, 'dominant_landcover']],
    'Distribution': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan], 
    'Link': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan], 
    'AIC': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan],
    'BIC': [np.nan, np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan, np.nan,np.nan]
    
    
}
model_compare_mean = pd.DataFrame(data)

In [30]:



for i, row in model_compare_mean.iterrows():
    if callable(row["func"]):  
        print(row['index_vars'])
        model_fits_mean = row["func"](df = fire3[row['row_mask']], y = row['y'],  x = row['x'], c = row['c'], opt_tag = row['opt_tag'], index_vars = row['index_vars'])
        model_fits_mean = model_fits_mean[model_fits_mean.BIC == model_fits_mean.BIC.min()]
        model_compare_mean.loc[i, 'Distribution'] = model_fits_mean.Family.iloc[0]
        model_compare_mean.loc[i, 'Link'] = model_fits_mean.Link_Function	.iloc[0]
        model_compare_mean.loc[i, 'AIC'] = model_fits_mean.AIC.iloc[0]
        model_compare_mean.loc[i, 'BIC'] = model_fits_mean.BIC.iloc[0]

/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress

['fireID', 't', 'GEOS-5.IMERGEARLY', 'FWI', 'field_response_type', 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling', 'GEOS5_IMERGEARLY_rolling', 'log_farea_diff_rolling', 'FWI_rolling_mean', 'farea_diff_rolling_mean', 'GEOS5_IMERGEARLY_rolling_mean']


/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:308: DomainWarning: The Identity link function does not respect the domain of the Gamma family.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-like

['fireID', 't', 'GEOS-5.IMERGEARLY', 'FWI', 'field_response_type', 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling', 'GEOS5_IMERGEARLY_rolling', 'log_farea_diff_rolling', 'FWI_rolling_mean', 'farea_diff_rolling_mean', 'GEOS5_IMERGEARLY_rolling_mean']
['fireID', 't', 'GEOS-5.IMERGEARLY', 'FWI', 'field_response_type', 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling', 'GEOS5_IMERGEARLY_rolling', 'log_farea_diff_rolling', 'FWI_rolling_mean', 'farea_diff_rolling_mean', 'GEOS5_IMERGEARLY_rolling_mean']


/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-like

...

/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress

['fireID', 't', 'GEOS-5.IMERGEARLY', 'FWI', 'field_response_type', 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling', 'GEOS5_IMERGEARLY_rolling', 'log_farea_diff_rolling', 'FWI_rolling_mean', 'farea_diff_rolling_mean', 'GEOS5_IMERGEARLY_rolling_mean', 'dominant_landcover']
['fireID', 't', 'GEOS-5.IMERGEARLY', 'FWI', 'field_response_type', 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling', 'GEOS5_IMERGEARLY_rolling', 'log_farea_diff_rolling', 'FWI_rolling_mean', 'farea_diff_rolling_mean', 'GEOS5_IMERGEARLY_rolling_mean', 'dominant_landcover']


/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:308: DomainWarning: The Identity link function does not respect the domain of the Gamma family.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-like

['fireID', 't', 'GEOS-5.IMERGEARLY', 'FWI', 'field_response_type', 'FWI_rolling', 'farea_diff_rolling', 'GEOS-5.IMERGEARLY_rolling', 'GEOS5_IMERGEARLY_rolling', 'log_farea_diff_rolling', 'FWI_rolling_mean', 'farea_diff_rolling_mean', 'GEOS5_IMERGEARLY_rolling_mean', 'dominant_landcover']


/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:308: DomainWarning: The Identity link function does not respect the domain of the Gamma family.
/srv/conda/envs/notebook/lib/python3.12/site-packages/statsmodels/genmod/generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-like

In [31]:
model_compare_mean

,Model,y,x,c,func,row_mask,opt_tag,index_vars,Distribution,Link,AIC,BIC
0,y ~ x:(Suppression Response),farea_diff_rolling_mean,FWI_rolling_mean,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5819.513198,-3134.488641
1,y ~ xIMERGE:(Suppression Response),farea_diff_rolling_mean,GEOS5_IMERGEARLY_rolling_mean,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,5812.455526,-3141.546313
2,y ~ x,farea_diff_rolling_mean,FWI_rolling_mean,NaN,<function run_stats_single at 0x7f6e00589e40>,0 False 1 True 2 False 3 ...,_single_model,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5873.567362,-3089.559365
3,y ~ xIMERGE,farea_diff_rolling_mean,GEOS5_IMERGEARLY_rolling_mean,NaN,<function run_stats_single at 0x7f6e00589e40>,0 False 1 True 2 False 3 ...,_single_model,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,5896.313031,-3066.813696
4,y ~ x:(Fuel Type),farea_diff_rolling_mean,FWI_rolling_mean,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5743.115392,-3201.761559
5,y ~ xIMERGE:(Fuel Type),farea_diff_rolling_mean,GEOS5_IMERGEARLY_rolling_mean,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,5722.886800,-3221.990151
6,yFuels N > 10 ~ xFuels N > 10:(Fuel Type),farea_diff_rolling_mean,FWI_rolling_mean,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5743.115392,-3201.761559
7,yFuels N > 10 ~ xIMERGEFuels N > 10:(Fuel Type),farea_diff_rolling_mean,GEOS5_IMERGEARLY_rolling_mean,dominant_landcover,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,5722.886800,-3221.990151
8,yFuels N > 10 ~ xFuels N > 10:(Suppression Res...,farea_diff_rolling_mean,FWI_rolling_mean,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Identity,5819.513198,-3134.488641
9,yFuels N > 10 ~ xIMERGEFuels N > 10:(Suppressi...,farea_diff_rolling_mean,GEOS5_IMERGEARLY_rolling_mean,field_response_type,<function run_stats at 0x7f6e2ed93060>,0 False 1 True 2 False 3 ...,fules_larger_10,"[fireID, t, GEOS-5.IMERGEARLY, FWI, field_resp...",NegativeBinomial,Log,5812.455526,-3141.546313
